# 03 — From opacity to expected muon counts

*Week 5*

This notebook converts the opacity map into expected muon counts. The
student-owned `expected_counts` function is stored in `code/student_analysis.py`,
so later notebooks do not depend on variables left in memory here.

The purpose of this notebook is to produce the **before-and-after pair**:
the true target (which we drew ourselves) and the reconstructed image that the
muons alone reveal.


In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive.*')
sys.path.insert(0, os.path.abspath('../code'))
import numpy as np
import matplotlib.pyplot as plt
import muography as mg
import project_config as cfg
import student_analysis as sa
print('ready')

ready


## The conversion

For each angular pixel:

`expected counts = transmitted flux × detector area × pixel solid angle × time`

The transmitted flux comes from `mg.transmitted_flux(opacity, theta)`. The
baseline detector area is 0.25 m². The pixel solid angle is approximated by
`mg.pixel_solid_angle(step_rad)`, which is the small-angle value
$\Omega \approx (\Delta\theta)^2$. The approximation is discussed in the
systematics section of notebook 05; for the official project result it is the
convention fixed by the supplied toolkit.


In [2]:
# Build the baseline target, its no-void twin, and the detector position.
target = mg.Pyramid(
    base=cfg.PYRAMID_BASE_M, height=cfg.PYRAMID_HEIGHT_M,
    density=cfg.ROCK_DENSITY_G_CM3, void_centre=cfg.VOID_CENTRE_M,
    void_radius=cfg.VOID_RADIUS_M, has_void=True,
)
target0 = target.without_void()
detector = np.array(cfg.DETECTOR_POSITION_M, dtype=float)

print('target with void  :', target.density_at(0, 0, 70), 'g/cm^3 inside the chamber')
print('twin without void :', target0.density_at(0, 0, 70), 'g/cm^3 at the same point')

target with void  : 0.0012 g/cm^3 inside the chamber
twin without void : 2.5 g/cm^3 at the same point


### YOUR TURN

Implement `expected_counts(...)` in `code/student_analysis.py`. It should:

1. call `opacity_map`,
2. compute the zenith angle for every viewing direction,
3. use `mg.transmitted_flux`,
4. multiply by area, pixel solid angle and exposure time, and
5. return `(counts, ax_deg, ay_deg)`.

The implementation is in the shared module, so notebooks 04 and 05 use exactly
the same function. The cell below is the reproducibility check: it must run
from a fresh kernel without first running notebook 02.


In [3]:
# After implementing expected_counts in code/student_analysis.py:
counts_void, ax_deg, ay_deg = sa.expected_counts(target, detector, days=30.0)
counts_solid, _, _ = sa.expected_counts(target0, detector, days=30.0)

print('30-day count range (with void):', counts_void.min(), counts_void.max())
print('30-day total counts (with void):', counts_void.sum())
print('30-day total counts (no void)  :', counts_solid.sum())
print('relative excess in total      :',
      f'{100*(counts_void.sum()-counts_solid.sum())/counts_solid.sum():.2f}%')

30-day count range (with void): 190.86679260174412 1196.8932172182604
30-day total counts (with void): 984874.6841419444
30-day total counts (no void)  : 981356.4941469305
relative excess in total      : 0.36%


### YOUR TURN — display it

Show the 30-day expected-count map. Can you see the chamber? The answer should
be that the overall pyramid shape dominates. That is why we need a no-chamber
reference.


In [4]:
extent = [ax_deg[0], ax_deg[-1], ay_deg[0], ay_deg[-1]]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
im = ax.imshow(counts_void, origin='lower', extent=extent, aspect='auto',
               cmap='viridis')
ax.set_xlabel('ax (degrees)')
ax.set_ylabel('ay (degrees)')
ax.set_title('Expected muon counts in 30 days (chamber present)')
fig.colorbar(im, ax=ax, label='expected counts per 2° pixel')
plt.tight_layout()
plt.savefig('../figures/08_expected_counts.png', dpi=300, bbox_inches='tight')
plt.show()

print('The image is dominated by the pyramid thickness:')
print('  thinnest direction: ', f'{counts_void.min():.2f} counts')
print('  thickest direction:', f'{counts_void.max():.2f} counts')
print('  ratio thick/thin  :', f'{counts_void.max()/counts_void.min():.2f}')

The image is dominated by the pyramid thickness:
  thinnest direction:  190.87 counts
  thickest direction: 1196.89 counts
  ratio thick/thin  : 6.27


## Divide out the target shape

Build the same expected-count map for `target.without_void()` and calculate:

`ratio = expected_with_void / expected_without_void`

Away from the chamber the ratio should be close to 1.0. The chamber should
appear as an excess above 1.0. This is Key result 3.1 of the handbook.


In [5]:
ratio = sa.ratio_map(counts_void, counts_solid)
signal_idx = sa.signal_pixel(counts_void, counts_solid)

fig, axes = plt.subplots(1, 2, figsize=(13, 5),
                         gridspec_kw={'width_ratios': [1.25, 1]})

im = axes[0].imshow(ratio, origin='lower', extent=extent, aspect='auto',
                    cmap='inferno', vmin=0.95, vmax=ratio.max())
axes[0].set_xlabel('ax (degrees)')
axes[0].set_ylabel('ay (degrees)')
axes[0].set_title('Clean ratio: with chamber / without chamber')
fig.colorbar(im, ax=axes[0], label='count ratio')

profile = ratio[:, np.argmin(np.abs(ax_deg))]
axes[1].plot(ay_deg, profile, 'o-', ms=3)
axes[1].axhline(1.0, color='k', lw=0.8, ls='--', label='no chamber')
axes[1].axvline(ay_deg[signal_idx[0]], color='C3', lw=0.8, ls=':',
                label='peak pixel')
axes[1].set_xlabel('ay (degrees)')
axes[1].set_ylabel('count ratio')
axes[1].set_title('Profile at ax = 0')
axes[1].legend(frameon=False)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/09_clean_ratio.png', dpi=300, bbox_inches='tight')
plt.show()

print('peak ratio              :', f'{ratio[signal_idx]:.4f}')
print('peak pixel (ax, ay)     :', f'({ax_deg[signal_idx[1]]:.0f}, {ay_deg[signal_idx[0]]:.0f}) degrees')
print('counts with / without   :', f'{counts_void[signal_idx]:.2f} / {counts_solid[signal_idx]:.2f}')
print('excess counts in 30 days:', f'{counts_void[signal_idx]-counts_solid[signal_idx]:.2f}')
print('median ratio away from chamber:',
      f'{np.median(ratio[np.abs(ay_deg) < 10]):.4f}')
print('expected significance at peak :',
      f'{mg.expected_significance(counts_void, counts_solid)[signal_idx]:.3f} sigma')

peak ratio              : 1.4758
peak pixel (ax, ay)     : (0, 30) degrees
counts with / without   : 366.79 / 248.53
excess counts in 30 days: 118.26
median ratio away from chamber: 1.0000
expected significance at peak : 7.502 sigma


### What the image shows

The raw expected-count map is a picture of the **pyramid**, not of the chamber.
The thinnest and thickest directions differ by a factor of about six, while the
chamber changes the count in a single pixel by about 48 per cent. A 48 per cent
effect hidden underneath a factor-two gradient is invisible by eye, which is
exactly the failure mode the handbook warns about.

Dividing by the no-void twin removes the shape. Away from the chamber the ratio
is 1.00 to within a fraction of a per cent. At the chamber it peaks at about
1.48: the missing 10 m of rock lets roughly half as many extra muons through.
This ratio map, not the raw-count map, is the **after** half of the
before-and-after pair.

The peak is at `ay ≈ 30°`, `ax = 0°`. That is the line of sight from the
detector at `(0, −40, 3)` m to the chamber centre at `(0, 0, 70)` m:

$$
\tan\theta = \frac{40}{70-3} = 0.597
\quad\Longrightarrow\quad
\theta \approx 30.8^\circ .
$$

The 2° angular grid samples this at 30°, one pixel from the exact centre. That
is why the peak ratio (1.48) is slightly below the ideal value one would get
for a ray through the centre: the ray misses the centre by about 1.3 m, so its
chord through the sphere is slightly shorter than 20 m.

The expected significance at the peak after 30 days is about **7.5σ**. The
smooth curve in notebook 04 will turn that into the headline exposure time.


### Why this ratio is the reconstruction

The raw count map mostly tells you the shape of the pyramid. The ratio removes
that known geometric effect and leaves the chamber contrast. This clean ratio
is the “after” image for the before-and-after pair.

Do not treat the clean ratio as noisy detector data; notebook 04 adds Poisson
fluctuations. The ratio is also not a full tomographic reconstruction: one
viewing direction becomes one pixel, exactly as in Alvarez's original pyramid
radiograph and the ScanPyramids figures. It is a **muon radiograph**, not a
three-dimensional density inversion.
